# Python Variable Scope & Namespaces (The LEGB Rule)

> **Topic:** Variable Scope & Namespaces | **Folder:** Data Structures & Algorithms

A **namespace** is a dictionary mapping variable names to their corresponding Python objects.
**Scope** defines the region of a program where a specific namespace is directly accessible.

Python follows the **LEGB Rule** to resolve variable names:
**L**ocal $\rightarrow$ **E**nclosing $\rightarrow$ **G**lobal $\rightarrow$ **B**uilt-in

Understanding scope is essential for managing state in **recursive algorithms**, **graph traversals**,
**closure factories**, and preventing unexpected side effects in algorithm implementations.

---

## Table of Contents
1. [Understanding Namespaces & Scope](#1.-Understanding-Namespaces-&-Scope)
2. [The LEGB Rule Overview](#2.-The-LEGB-Rule-Overview)
3. [Local Scope (L)](#3.-Local-Scope-(L))
4. [Enclosing Scope (E) & Closures](#4.-Enclosing-Scope-(E)-&-Closures)
5. [Global Scope (G) & the `global` Keyword](#5.-Global-Scope-(G)-&-the-`global`-Keyword)
6. [The `nonlocal` Keyword in Algorithmic Helpers](#6.-The-`nonlocal`-Keyword-in-Algorithmic-Helpers)
7. [Built-in Scope (B) & Avoiding Name Shadowing](#7.-Built-in-Scope-(B)-&-Avoiding-Name-Shadowing)
8. [Scope Isolation: Loops vs. Comprehensions](#8.-Scope-Isolation:-Loops-vs.-Comprehensions)
9. [Mutable vs. Immutable Global State Traps](#9.-Mutable-vs.-Immutable-Global-State-Traps)
10. [Quick Reference Card](#10.-Quick-Reference-Card)


---
## 1. Understanding Namespaces & Scope

In Python, everything is an object. A variable is simply a **name (reference)** pointing to an object.
Python manages these name-to-object bindings using dictionary-like structures called **namespaces**.

You can inspect current namespaces using built-in functions:
- `locals()`: Returns a dictionary of the current local namespace.
- `globals()`: Returns a dictionary of the current global namespace.


In [ ]:
# Inspecting globals()
GLOBAL_VAR = "I am global"

def demo_func():
    local_var = "I am local"
    print("Inside demo_func() locals():", locals())

demo_func()
print("GLOBAL_VAR in globals()?", "GLOBAL_VAR" in globals())


---
## 2. The LEGB Rule Overview

When you reference a variable `x`, Python searches namespaces in this exact sequence:

```
  [ L ]  Local       --> Names defined inside the current function/lambda
    │
  [ E ]  Enclosing   --> Names in enclosing functions (outer to inner)
    │
  [ G ]  Global      --> Names defined at top-level of module
    │
  [ B ]  Built-in    --> Pre-defined names in python builtins (len, sum, range...)
```

If `x` is not found in any of these four scopes, Python raises a `NameError`.


In [ ]:
# LEGB Lookup Demonstration
x = "[G] Global x"

def outer():
    x = "[E] Enclosing x"
    def inner():
        x = "[L] Local x"
        print("Inner sees :", x)  # Finds Local first
    inner()
    print("Outer sees :", x)    # Finds Enclosing

outer()
print("Global sees:", x)        # Finds Global


---
## 3. Local Scope (L)

Variables declared inside a function belong to that function's **Local Scope**.
They are created when the function is invoked and destroyed when the function returns.


In [ ]:
def calculate_area(radius):
    pi = 3.14159                  # Local variable
    area = pi * (radius ** 2)     # Local variable
    return area

print("Area:", calculate_area(5))

# Accessing local variable outside function raises NameError
try:
    print(pi)
except NameError as e:
    print(f"Caught NameError: {e}")


---
## 4. Enclosing Scope (E) & Closures

Enclosing (or Nonlocal) scope exists in nested functions.
An inner function can read variables defined in the outer enclosing function.


In [ ]:
def make_counter(start_value=0):
    # Enclosing scope variable
    count = start_value
    
    def get_count():
        return f"Current count: {count}"  # Reads from enclosing scope
    
    return get_count

c1 = make_counter(10)
print(c1())


---
## 5. Global Scope (G) & the `global` Keyword

To **rebind (modify)** a module-level global variable from inside a function,
you must explicitly declare it using the `global` keyword.


In [ ]:
execution_count = 0

def run_algorithm():
    global execution_count       # Declare intent to modify global variable
    execution_count += 1
    print(f"Algorithm executed {execution_count} time(s).")

run_algorithm()
run_algorithm()
print(f"Final global count: {execution_count}")


---
## 6. The `nonlocal` Keyword in Algorithmic Helpers

In nested functions (such as DFS helper functions in tree/graph algorithms),
the `nonlocal` keyword allows modifying variables in the outer **Enclosing scope**.


In [ ]:
# Binary Tree Node
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Finding Maximum Depth of Binary Tree using nonlocal helper
def maxDepth(root):
    max_d = 0                   # Variable in Enclosing scope
    
    def dfs(node, current_depth):
        nonlocal max_d          # Allows updating max_d from inside DFS helper
        if not node: return
        if current_depth > max_d:
            max_d = current_depth
        dfs(node.left, current_depth + 1)
        dfs(node.right, current_depth + 1)
        
    dfs(root, 1)
    return max_d

# Tree: 1 -> (2, 3 -> (4))
tree = TreeNode(1, TreeNode(2), TreeNode(3, None, TreeNode(4)))
print(f"Max Tree Depth: {maxDepth(tree)}")


---
## 7. Built-in Scope (B) & Avoiding Name Shadowing

Python's Built-in scope contains functions like `len()`, `max()`, `sum()`, `list`, `dict`.
Assigning a local/global variable with a built-in name **shadows** the built-in function,
causing unexpected errors when calling it later!


In [ ]:
# Shadowing Demonstration & Fix
numbers = [1, 2, 3, 4, 5]

# BAD: Shadowing built-in sum()
sum = 100   # Now 'sum' is an integer, not the built-in function!
print(f"Shadowed variable sum = {sum}")

try:
    total = sum(numbers)  # TypeError: 'int' object is not callable
except TypeError as e:
    print(f"Caught error due to shadowing: {e}")

# FIX: Restore built-in sum by deleting local/global shadow binding
del sum
print("Restored built-in sum(numbers) =", sum(numbers))


---
## 8. Scope Isolation: Loops vs. Comprehensions

- `for` loops **do NOT create a new scope** in Python! Loop variables leak into the surrounding scope.
- List/Set/Dict **comprehensions create their own isolated local scope**, keeping loop variables clean.


In [ ]:
# 1. Loop variable leakage
for k in range(5):
    pass
print(f"Variable 'k' leaked outside for loop: k = {k}")

# 2. Comprehension scope isolation
squares = [m ** 2 for m in range(5)]
print("Comprehension result:", squares)
try:
    print(m)
except NameError as e:
    print(f"Variable 'm' did NOT leak outside comprehension: {e}")


---
## 9. Mutable vs. Immutable Global State Traps

- Rebinding an **immutable** object (`int`, `str`, `tuple`) inside a function creates a **new local variable** unless `global` is declared.
- Mutating a **mutable** object (`list`, `dict`, `set`) inside a function **modifies the global object directly** without needing `global`!


In [ ]:
# Global mutable list
results_list = []

def append_result(val):
    results_list.append(val)  # Mutates existing list in place without 'global'

append_result(10)
append_result(20)
print("Mutated global list:", results_list)

# Rebinding mutable object requires 'global'
def reset_results():
    global results_list
    results_list = []         # Rebinding requires 'global'

reset_results()
print("Reset global list:", results_list)


---
## 10. Quick Reference Card


In [ ]:
# ==================================================================
# PYTHON VARIABLE SCOPE (LEGB) – QUICK REFERENCE
# ==================================================================

# L: Local    -> Defined inside current function
# E: Enclosing-> Defined in outer nested function
# G: Global   -> Top-level of module
# B: Built-in -> Python built-in names (len, range, int...)

# Rebinding Keywords:
# - 'global x'   : Allows rebinding module-level variable x inside function
# - 'nonlocal x' : Allows rebinding enclosing function variable x inside nested helper

def outer_scope():
    counter = 0
    def inner_scope():
        nonlocal counter
        counter += 1
        return counter
    return inner_scope

inc = outer_scope()
print("nonlocal counter:", inc(), inc())


---
## Summary

| Scope Level | Search Order | Description | Keyword to Rebind |
|-------------|--------------|-------------|--------------------|
| **Local (L)** | 1st | Variables declared inside function | *(default)* |
| **Enclosing (E)** | 2nd | Outer function variables in nested structures | `nonlocal` |
| **Global (G)** | 3rd | Module-level variables | `global` |
| **Built-in (B)** | 4th | Pre-defined Python names | Do not shadow! |

---
*Next up: **Object Oriented Programming (Classes, Methods & Inheritance)***
